In [0]:
import re
import requests
import os
from pyspark.sql.functions import lit, current_timestamp,split

In [0]:
CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
VOLUME = dbutils.widgets.get("VOLUME")
CITIES = dbutils.widgets.get("CITIES")
GET_DATA_URL = dbutils.widgets.get("GET_DATA_URL")
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
COUNTRY = dbutils.widgets.get("COUNTRY")
city_refined = re.sub(r"[^a-z0-9]+", "_", CITIES.lower()).strip("-")
city_volume_path = f"{VOLUME_PATH}/{city_refined}"
ingestion_table_name =f"{CATALOG}.{SCHEMA}.ingestion_control"

In [0]:
def local_variables(iterator):
    city_name = iterator.path.split("/")[-2].replace("_", "")
    schema_name = iterator.path.split("/")[-3].replace("raw", "bronze")
    table_name = iterator.path.split("/")[-1].replace(".", "_")
    full_table_name = f"{CATALOG}.{schema_name}.{city_name}_{table_name}"
    dimensions_group = full_table_name.split(".")[-1].split("_")[1]
    watermark_column_name = "load_ts"
    watermark_updated_ts = current_timestamp()
    execution_ts = current_timestamp()
    src_table_vw_name = "src_table_vw"

    return [
        {
            "city_name": city_name,
            "schema_name": schema_name,
            "table_name": table_name,
            "full_table_name": full_table_name,
        },
        {
            "object_name": full_table_name,
            "dimensions_group": dimensions_group,
            "watermark_column_name": watermark_column_name,
            "watermark_updated_ts": watermark_updated_ts,
            "execution_ts": execution_ts,
            "src_table_vw_name": src_table_vw_name,
        },
    ]

In [0]:
def get_page_html(url: str = GET_DATA_URL) -> str:
    """Fetch the Inside Airbnb data page HTML."""
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    return resp.text

In [0]:
def find_city_links(page_html: str, city: str) -> list:
    """Return the current data.insideairbnb.com download links from the city's section of the page.

    Each city has a section: <h3>City, State, Country</h3> followed by a table of
    file links. The section runs until the archived-data toggle / next city heading.
    """
    for m in re.finditer(r"<h3>([^<]+)</h3>", page_html, flags=re.IGNORECASE):
        if city.lower() not in m.group(1).lower():
            continue
        candidates = [
            x
            for x in (
                page_html.find("<h3", m.end()),
                page_html.find('class="showArchivedData"', m.end()),
            )
            if x != -1
        ]
        section_end = min(candidates) if candidates else len(page_html)
        return re.findall(r'href="(https://data\.insideairbnb\.com/[^"]+)"', page_html[m.end() : section_end])
    return []

In [0]:
def save_to_volume(url: str, dest: str) -> None:
    """Stream a remote file directly into the UC volume, overwriting any existing copy."""
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with requests.get(url, stream=True, timeout=600) as resp:
        resp.raise_for_status()
        with open(dest, "wb") as f:  # 'wb' overwrites any previous file
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)

In [0]:
# def download_city_files(city: str, page_html: str) -> list:
#     """Download every file currently listed for a city into the volume, overwriting existing copies.

#     Returns a list of (city, file, source_url, bytes_written) tuples.
#     """
#     links = find_city_links(page_html, city)
#     if not links:
#         return [(city, "NOT FOUND ON PAGE", "", 0)]

#     slug = re.sub(r"[^a-z0-9]+", "_", city.lower()).strip("-")
#     results = []
#     for url in links:
#         name = url.rstrip("/").split("/")[-1]
#         dest = f"{VOLUME_PATH}/{slug}/{name}"
#         save_to_volume(url, dest)
#         results.append((city, name, url, os.path.getsize(dest)))
#     return results

In [0]:
def download_city_files_refactoring(city: str, page_html: str):
    """Download every file currently listed for a city into the volume, overwriting existing copies.
    
    Returns a list of (city, file, source_url, bytes_written) tuples.
    """
    try:
        links = find_city_links(page_html, city)
        if not links:
            return f"city page not found for {city}"
        
        #slug = re.sub(r"[^a-z0-9]+", "_", city.lower()).strip("-")
        slug = city.lower().replace(" ","_")
        for url in links:
            name = url.rstrip("/").split("/")[-1]
            dest = f"{VOLUME_PATH}/{slug}/{name}"
            save_to_volume(url, dest)
    except Exception as e:
        error = str(e)
        return f"{error}"

In [0]:
# for i in dbutils.fs.ls(city_volume_path.replace('"','')):
#     print(i.path)

dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/calendar.csv.gz
dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/listings.csv
dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/listings.csv.gz
dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/neighbourhoods.csv
dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/neighbourhoods.geojson
dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/reviews.csv
dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/reviews.csv.gz


In [0]:
# def list_raw_files(volume_path: str = city_volume_path) -> list:
#     #volume_path =f"{VOLUME_PATH}/{CITIES}"
#     """Return absolute paths of all data files currently in the volume."""
#     files = []
#     for root, _, names in os.walk(volume_path):
#         for name in names:
#             files.append(os.path.join(root, name))
#     return files

In [0]:
# def load_file_to_table(src: str, catalog: str = CATALOG, schema: str = SCHEMA) -> str:
#     """Load one raw file from the volume into a Delta table (overwrite mode).

#     Table naming: <city-slug>_<file-stem>, e.g. new-york-city_listings -> tanmay_projects.project_1.new-york-city_listings
#     """
#     #volume_path =f"{VOLUME_PATH}/{CITIES}"
#     rel = os.path.relpath(src, VOLUME_PATH)  # e.g. "new-york/listings.csv"
#     city_slug, filename = rel.split(os.sep, 1)
#     city_slug = city_slug.replace("_","")
#     stem = os.path.splitext(filename)[0].replace(".", "_")
#     if filename.endswith(".geojson"):  # avoid colliding with the neighbourhoods.csv table
#         stem += "_geojson"
#     table_name = f"{city_slug}_{stem}".replace("-", "_")
#     full_table = f"{catalog}.{schema}.{table_name}"
    
#     if filename.endswith(".csv") or filename.endswith(".csv.gz"):
#         df = (
#             spark.read.format("csv")
#             .option("header", True)
#             .option("inferSchema", True)
#             .option("multiLine", True)
#             .load(src)
#         )
#         df.select("*",lit(COUNTRY).alias("Country"),current_timestamp().alias("load_ts")).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(full_table)
#     elif filename.endswith(".geojson"):
#         df = spark.read.format("json").option("multiLine", True).load(src)
#         df.select("*",lit(COUNTRY).alias("Country"),current_timestamp().alias("load_ts")).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(full_table)
#     else:
#         print(f"Skipping unsupported file type: {rel}")
#         return None
#     return full_table

In [0]:
# def get_table_name(src: str, catalog: str = CATALOG, schema: str = SCHEMA)-> str:
#     rel = os.path.relpath(src, VOLUME_PATH)
#     city_slug, filename = rel.split(os.sep, 1)
#     stem = os.path.splitext(filename)[0].replace(".", "_")
#     if filename.endswith(".geojson"):
#         stem += "_geojson"
#     table_name = f"{city_slug}_{stem}".replace("-", "_")
#     full_table = f"{catalog}.{schema}.{table_name}"
#     return full_table

In [0]:
# def load_file_to_table_refactoring(src: str, catalog: str = CATALOG, schema: str = SCHEMA) -> str:
#     rel = os.path.relpath(src, VOLUME_PATH)
#     city_slug, filename = rel.split(os.sep, 1)
#     stem = os.path.splitext(filename)[0].replace(".", "_")
#     if filename.endswith(".geojson"):
#         stem += "_geojson"
#     table_name = f"{city_slug}_{stem}".replace("-", "_")
#     full_table = f"{catalog}.{schema}.{table_name}"
#     if filename.endswith(".csv") or filename.endswith(".csv.gz"):
#         df = (
#             spark.read.format("csv")
#             .option("header", True)
#             .option("inferSchema", True)
#             .option("multiLine", True)
#             .load(src)
#         )
        
#         df.select("*", lit(COUNTRY).alias("Country"), current_timestamp().alias("load_ts")).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(full_table)
#         ingestion_control_refactoring(full_table)
#     elif filename.endswith(".geojson"):
#         df = spark.read.format("json").option("multiLine", True).load(src)
#         df.select("*", lit(COUNTRY).alias("Country"), current_timestamp().alias("load_ts")).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(full_table)
#         ingestion_control_refactoring(full_table)
#     else:
#         print(f"Skipping unsupported file type: {rel}")
#         return None
#     return full_table

In [0]:
# def ingestion_control_refactoring(full_table):
#     object_name =full_table
#     dimension_group = full_table.split('.')[-1].split('_')[-2]
#     watermark_column_name = 'load_ts'
#     watermark_updated_ts = current_timestamp()
#     error_message = None
#     status = None
#     execution_ts = current_timestamp()
#     src_table_vw_name = 'src_table_vw'
#     spark.createDataFrame(
#         [(object_name, dimension_group, watermark_column_name, None, error_message, status, None)],
#         "object_name STRING, dimensions_group STRING, watermark_column_name STRING, watermark_updated_ts TIMESTAMP, error_message STRING, status STRING, execution_ts TIMESTAMP"
#     ).withColumn("watermark_updated_ts",current_timestamp()).withColumn("execution_ts",current_timestamp()).createOrReplaceTempView(src_table_vw_name)
#     spark.sql(f"""merge into {table_name} IC
#                 using {src_table_vw_name} SV
#                 on IC.object_name = SV.object_name and IC.execution_ts = SV.execution_ts
#                 when matched then update set IC.watermark_updated_ts = SV.watermark_updated_ts, IC.status = SV.status, IC.error_message = SV.error_message
#                 when not matched then insert
#                 (object_name, dimensions_group, watermark_column_name, watermark_updated_ts, error_message, status, execution_ts)
#                 values
#                 (SV.object_name, SV.dimensions_group, SV.watermark_column_name, SV.watermark_updated_ts, SV.error_message, SV.status, SV.execution_ts)""")

In [0]:
def ingestion_control_refactoring(metadata_variables,error_message,status):
    
    # object_name =full_table
    # dimensions_group = full_table.split('.')[-1].split('_')[1]
    # watermark_column_name = 'load_ts'
    # watermark_updated_ts = current_timestamp()
    error_message = error_message
    status = status
    # execution_ts = current_timestamp()
    # src_table_vw_name = 'src_table_vw'

    df = spark.range(1).select(
    lit(metadata_variables['object_name']).alias('object_name'),
    lit(metadata_variables['dimensions_group']).alias('dimensions_group'),
    lit(metadata_variables['watermark_column_name']).alias('watermark_column_name'),
    metadata_variables['watermark_updated_ts'].alias('watermark_updated_ts'),
    lit(error_message).alias('error_message'),
    lit(status).cast("string").alias("status"),
    metadata_variables['execution_ts'].alias("execution_ts")
    ).createOrReplaceTempView(metadata_variables['src_table_vw_name'])

    spark.sql(f"""merge into {ingestion_table_name} IC
                using {metadata_variables['src_table_vw_name']} SV
                on IC.object_name = SV.object_name and IC.execution_ts = SV.execution_ts
                when matched then update set IC.watermark_updated_ts = SV.watermark_updated_ts, IC.status = SV.status, IC.error_message = SV.error_message
                when not matched then insert
                (object_name, dimensions_group, watermark_column_name, watermark_updated_ts, error_message, status, execution_ts)
                values
                (SV.object_name, SV.dimensions_group, SV.watermark_column_name, SV.watermark_updated_ts, SV.error_message, SV.status, SV.execution_ts)""")